# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink and mapping de-identified events to OMOP CDM v5.4.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, to_json, struct
from splink.spark.linker import SparkLinker
from splink.spark.blocking_rule_library import block_on
import splink.spark.comparison_library as cl

spark = SparkSession.builder \
    .appName("Day05_OMOP_Linkage") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# 1. Probabilistic Linkage with Splink
# Simulating records from different EHR systems
data = [
    ("1", "John", "Doe", "1980-01-01", "M"),
    ("2", "Jon", "Doe", "1980-01-01", "M"), # Likely same person
    ("3", "Jane", "Smith", "1990-05-15", "F")
]

df = spark.createDataFrame(data, ["unique_id", "first_name", "last_name", "dob", "gender"])

settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        block_on("dob"),
        block_on("gender")
    ],
    "comparisons": [
        cl.levenshtein_at_thresholds("first_name", 2),
        cl.exact_match("last_name"),
        cl.exact_match("dob"),
        cl.exact_match("gender")
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(df, settings, spark=spark)
linker.estimate_u_using_random_sampling(max_pairs=1e6)

training_blocking_rule = block_on("first_name", "last_name")
linker.estimate_parameters_using_expectation_maximisation(training_blocking_rule)

df_predictions = linker.predict(threshold_match_probability=0.9)
print("Linkage Predictions (Probabilistic matching found John & Jon):")
df_predictions.as_pandas_dataframe()[["unique_id_l", "first_name_l", "unique_id_r", "first_name_r", "match_probability"]]

In [ ]:
# 2. OMOP Transformation
bronze_data = [
    ("payload_001", "{"encounter_type": "inpatient", "diagnosis": "Nausea"}")
]
df_bronze = spark.createDataFrame(bronze_data, ["payload_id", "raw_payload"])

omop_condition_df = df_bronze.select(
    col("payload_id").alias("condition_occurrence_id"),
    lit("person_123").alias("person_id"),
    lit(31967).alias("condition_concept_id"),
    current_timestamp().alias("condition_start_date"),
    lit(32020).alias("condition_type_concept_id"),
    lit("Nausea").alias("condition_source_value")
)

print("\nOMOP CDM v5.4 condition_occurrence format:")
omop_condition_df.show(truncate=False)